In [ ]:
import nest_asyncio
from fastapi import FastAPI, HTTPException, Query
from pydantic import BaseModel, Field
from typing import List, Optional
from datetime import datetime
import uvicorn

nest_asyncio.apply()

app = FastAPI(title = "Blog API", description="API de gestion d'articles avec swagger", version="1.0.0")

class Article(BaseModel):
    id:Optional[int] =None
    titre: str = Field(...,min_length=3, example= "Mon premier article")
    contenu: str = Field(...,example= "contenu detailler de l'article")
    auteur: str = Field(...,example= "Paul Alin")
    date: str = Field(default_factory=lambda: datetime.now().strftime("%d/%m/%Y"))
    categorie: str
    tags: List[str] = []

    db_articles = []
    id_counter = 1

    # developpement des Endpoints(CRUD)
#1. creer un article (post)
@app.post("/api/articles", status_code=201)
async def create_article(article: Article, db_articles=None):
    global id_counter
    article.id = id_counter
    db_articles.append(article.dict())
    id_counter += 1
    return {"message": "Article créé avec succès", "id": article.id}
# 2. Lire tous les articles avec filtres (GET)
@app.get("/api/articles", response_model=List[Article])
async def get_articles(categorie: Optional[str] = None, auteur: Optional[str] = None):
    results = db_articles
    if categorie:
        results = [a for a in results if a['categorie'].lower() == categorie.lower()]
    if auteur:
        results = [a for a in results if a['auteur'].lower() == auteur.lower()]
    return results

# 3. Rechercher un article (Recherche textuelle)
@app.get("/api/articles/search")
async def search_articles(query: str = Query(..., min_length=1)):
    results = [a for a in db_articles if query.lower() in a['titre'].lower() or query.lower() in a['contenu'].lower()]
    return results

# 4. Lire un article unique (GET by ID)
@app.get("/api/articles/{article_id}")
async def get_article(article_id: int):
    article = next((a for a in db_articles if a['id'] == article_id), None)
    if not article:
        raise HTTPException(status_code=404, detail="Article non trouvé")
    return article
# 5. Modifier un article (PUT)
@app.put("/api/articles/{article_id}")
async def update_article(article_id: int, updated_data: Article):
    for index, article in enumerate(db_articles):
        if article['id'] == article_id:
            updated_dict = updated_data.dict()
            updated_dict['id'] = article_id  # Garder le même ID
            db_articles[index] = updated_dict
            return {"message": "Article mis à jour"}
    raise HTTPException(status_code=404, detail="Article non trouvé")

# 6. Supprimer un article (DELETE)
@app.delete("/api/articles/{article_id}")
async def delete_article(article_id: int):
    global db_articles
    initial_length = len(db_articles)
    db_articles = [a for a in db_articles if a['id'] != article_id]
    if len(db_articles) == initial_length:
        raise HTTPException(status_code=404, detail="Article non trouvé")
    return {"message": "Article supprimé avec succès"}

if __name__ == "__main__":
    uvicorn.run(app, host="127.0.0.1", port=8000)